# 💳 Loan Eligibility & Risk Prediction

**AI23521 – Build and Deployment of Machine Learning Applications**

This notebook demonstrates preprocessing, Decision Tree, SVM, Random Forest, model evaluation, GridSearchCV, cross-validation, and model saving.

> Educational project only. It should not be used for real lending decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import joblib

np.random.seed(42)
print('Libraries imported successfully!')

## 1. Create the Dataset

In [ ]:
n = 1000

df = pd.DataFrame({
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Married': np.random.choice(['Yes', 'No'], n),
    'Dependents': np.random.choice(['0', '1', '2', '3+'], n),
    'Education': np.random.choice(['Graduate', 'Not Graduate'], n, p=[0.75, 0.25]),
    'Self_Employed': np.random.choice(['Yes', 'No'], n, p=[0.15, 0.85]),
    'ApplicantIncome': np.random.randint(1500, 15000, n),
    'CoapplicantIncome': np.random.randint(0, 8000, n),
    'LoanAmount': np.random.randint(40, 500, n),
    'Loan_Amount_Term': np.random.choice([180, 240, 300, 360], n),
    'Credit_History': np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'Property_Area': np.random.choice(['Urban', 'Semiurban', 'Rural'], n)
})

# Create an educational target variable
income = df['ApplicantIncome'] + df['CoapplicantIncome']
score = (3 * df['Credit_History']
         + (income > 5000).astype(int)
         + (income > 9000).astype(int)
         + (df['LoanAmount'] < 250).astype(int)
         + (df['Education'] == 'Graduate').astype(int))

noise = np.random.normal(0, 0.8, n)
df['Loan_Status'] = ((score + noise) >= 4).astype(int)

# Introduce a few missing values for preprocessing demonstration
for col in ['Gender', 'Self_Employed', 'LoanAmount']:
    idx = np.random.choice(df.index, size=20, replace=False)
    df.loc[idx, col] = np.nan

df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['Loan_Status'].value_counts())

## 2. Data Preprocessing

In [ ]:
categorical_features = [
    'Gender', 'Married', 'Dependents', 'Education',
    'Self_Employed', 'Property_Area'
]

numeric_features = [
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
    'Loan_Amount_Term', 'Credit_History'
]

# Fill missing values
for col in categorical_features:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))

## 3. Train Decision Tree, SVM and Random Forest

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = []
trained_models = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    trained_models[name] = pipeline

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1 Score': f1_score(y_test, pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

## 4. Compare Model Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['Accuracy'])
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.show()

## 5. Confusion Matrices

In [ ]:
for name, pipeline in trained_models.items():
    pred = pipeline.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f'{name} - Confusion Matrix')
    plt.show()

## 6. Hyperparameter Tuning using GridSearchCV

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 8, 12],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print('Best Parameters:', grid_search.best_params_)
print('Best CV F1 Score:', round(grid_search.best_score_, 4))

## 7. Evaluate the Tuned Model

In [ ]:
best_pred = best_model.predict(X_test)

print('Accuracy :', round(accuracy_score(y_test, best_pred), 4))
print('Precision:', round(precision_score(y_test, best_pred), 4))
print('Recall   :', round(recall_score(y_test, best_pred), 4))
print('F1 Score :', round(f1_score(y_test, best_pred), 4))

ConfusionMatrixDisplay.from_predictions(y_test, best_pred)
plt.title('Tuned Random Forest - Confusion Matrix')
plt.show()

## 8. 5-Fold Cross-Validation

In [ ]:
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='accuracy')
print('CV Scores:', np.round(cv_scores, 4))
print('Mean CV Accuracy:', round(cv_scores.mean(), 4))

## 9. Save the Trained Model

In [ ]:
joblib.dump(best_model, 'loan_model.pkl')
print('Model saved as loan_model.pkl')

## 10. Test with a New Loan Applicant

In [ ]:
new_applicant = pd.DataFrame([{
    'Gender': 'Female',
    'Married': 'No',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 7500,
    'CoapplicantIncome': 2500,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1,
    'Property_Area': 'Urban'
}])

prediction = best_model.predict(new_applicant)[0]
probability = best_model.predict_proba(new_applicant)[0][1]

print('Loan Eligibility:', 'Eligible' if prediction == 1 else 'Not Eligible')
print('Eligibility Probability:', f'{probability * 100:.2f}%')

## Conclusion

This project demonstrates an end-to-end machine-learning workflow for educational loan eligibility prediction. It preprocesses mixed data, compares multiple classification algorithms, tunes a Random Forest model using GridSearchCV, evaluates it with cross-validation, and saves the trained model for later Flask deployment.